# Диана: мектеп жылыжайын суару — 200 жазба

Деректер → тексеру → 160/40 бөлу → нормалау → оқыту → бағалау → Streamlit.

Бұл жасанды оқу деректері, нақты датчик өлшемдері емес. Нәтиже шартты ережемен белгіленген: топырақ <30%, немесе топырақ <45%, температура >28°C және ауа ылғалдылығы <55%. Жоғары баға нақты жылыжайда тиімділікті дәлелдемейді.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import joblib, sklearn, sys
folder = Path("Diana_Greenhouse_200")
folder.mkdir(exist_ok=True)

## 1. 200 деректі жүктеу
CSV ноутбукке енгізілген: бөлек жүктеу және интернет қажет емес. Жинақ Амирханның деректерінен бөлек құрастырылған; seed=2026.

In [ ]:
from io import StringIO
csv_text = 'temperature_c,air_humidity_pct,soil_moisture_pct,irrigation_needed\n18.6,76.5,25.3,1\n27.8,69.1,27.8,1\n24.3,42.4,37.7,0\n22.4,74.7,12.4,1\n22.1,76.3,13.8,1\n30.8,76.2,56.4,0\n33.1,34.7,32.2,1\n18.5,42.7,60.4,0\n28.1,83.7,15.2,1\n21.0,82.8,61.8,0\n34.3,42.3,25.0,1\n33.4,63.1,70.9,0\n27.7,45.8,73.5,0\n30.1,55.0,58.3,0\n25.3,31.3,37.6,0\n31.5,46.9,11.8,1\n24.0,46.2,63.3,0\n21.8,85.1,49.4,0\n20.6,65.3,15.8,1\n19.5,40.7,50.0,0\n25.5,55.7,10.4,1\n23.6,59.6,41.6,0\n28.3,68.2,52.9,0\n15.3,61.9,66.2,0\n24.0,55.3,30.3,0\n22.3,48.6,26.2,1\n18.9,64.5,33.2,0\n26.9,88.5,14.1,1\n23.7,63.2,75.2,0\n21.0,83.2,71.6,0\n19.2,30.9,52.1,0\n32.5,42.2,15.2,1\n30.9,81.9,15.9,1\n27.1,64.2,75.2,0\n21.9,59.3,18.7,1\n33.9,52.1,20.2,1\n26.3,47.7,65.0,0\n23.7,70.7,31.1,0\n33.0,51.9,38.8,1\n21.4,56.1,38.7,0\n28.9,76.6,59.0,0\n21.3,86.8,43.1,0\n20.2,89.6,66.9,0\n29.0,59.1,44.7,0\n19.6,38.3,30.3,0\n24.9,31.5,23.6,1\n26.6,59.7,62.9,0\n18.8,34.6,25.4,1\n29.6,37.1,52.6,0\n26.0,44.1,52.1,0\n27.4,89.5,77.4,0\n22.4,39.7,16.3,1\n23.4,36.6,24.5,1\n24.9,85.0,56.3,0\n24.4,67.3,33.2,0\n28.5,74.4,19.0,1\n26.5,51.3,72.7,0\n23.3,86.0,61.2,0\n15.0,88.6,25.6,1\n30.9,63.5,16.7,1\n25.4,89.2,69.0,0\n21.5,60.3,49.9,0\n25.0,56.4,75.1,0\n16.9,77.1,46.2,0\n33.1,68.7,26.2,1\n34.8,42.0,19.3,1\n16.2,60.9,39.3,0\n22.2,31.0,57.9,0\n29.6,82.5,10.3,1\n21.3,62.5,37.7,0\n26.3,32.8,38.3,0\n23.3,86.3,57.0,0\n30.5,90.0,23.6,1\n34.2,74.2,71.3,0\n32.8,55.9,32.8,0\n27.4,65.5,67.9,0\n18.2,30.6,67.9,0\n33.9,36.5,71.9,0\n15.5,49.9,28.1,1\n21.0,54.7,68.8,0\n20.6,75.4,26.1,1\n28.4,40.2,38.0,1\n24.7,37.0,22.4,1\n16.9,71.8,43.9,0\n15.3,83.3,15.3,1\n27.1,82.8,77.0,0\n24.8,74.5,10.9,1\n27.0,89.0,44.8,0\n26.3,61.3,68.7,0\n32.8,76.5,19.3,1\n33.4,55.0,51.8,0\n18.8,51.2,47.2,0\n33.8,84.6,63.0,0\n30.8,55.8,58.9,0\n27.8,74.8,67.8,0\n28.2,59.8,65.5,0\n25.9,36.7,50.8,0\n33.3,37.6,34.5,1\n19.7,60.6,17.5,1\n27.0,72.5,21.5,1\n31.3,74.5,71.0,0\n17.7,50.1,28.7,1\n27.3,30.6,72.6,0\n23.1,62.0,31.4,0\n30.3,41.4,14.0,1\n16.4,49.2,30.3,0\n27.1,64.0,36.9,0\n32.1,53.5,17.3,1\n27.6,57.5,40.1,0\n21.4,33.0,65.9,0\n28.1,41.0,39.0,1\n21.8,80.5,44.6,0\n28.4,60.7,79.9,0\n17.6,83.1,25.1,1\n20.6,81.4,48.1,0\n15.7,41.6,79.1,0\n16.7,54.0,44.7,0\n26.1,61.5,51.4,0\n19.6,35.9,48.3,0\n25.3,35.1,18.1,1\n28.2,80.5,60.1,0\n32.6,79.6,73.8,0\n22.1,53.0,74.9,0\n21.4,72.4,51.1,0\n21.3,61.1,34.7,0\n17.4,74.9,57.3,0\n28.3,48.9,76.1,0\n31.4,58.4,78.5,0\n24.5,83.6,29.8,1\n26.2,79.0,75.5,0\n34.2,48.6,61.1,0\n29.0,82.9,71.0,0\n29.9,53.9,21.7,1\n32.8,70.9,78.3,0\n19.3,72.7,62.7,0\n28.0,54.1,21.7,1\n29.3,79.6,47.8,0\n31.8,57.9,17.8,1\n21.9,87.6,60.7,0\n28.8,86.2,51.7,0\n27.1,84.0,50.5,0\n23.2,84.5,69.3,0\n17.0,38.8,63.9,0\n22.1,68.9,16.4,1\n34.3,74.5,63.3,0\n19.3,61.7,47.9,0\n17.3,61.8,23.2,1\n22.4,42.1,34.3,0\n17.0,39.6,45.6,0\n20.3,77.0,70.1,0\n33.3,39.0,71.7,0\n21.4,73.7,17.1,1\n26.7,34.6,41.0,0\n19.8,54.7,69.0,0\n28.4,31.4,52.2,0\n17.6,69.6,69.9,0\n30.8,47.6,51.8,0\n23.5,58.1,25.6,1\n18.7,47.2,45.2,0\n30.2,60.9,46.1,0\n26.1,69.5,72.6,0\n23.8,40.0,34.9,0\n19.8,76.6,26.4,1\n30.6,77.0,62.4,0\n22.3,86.5,39.4,0\n15.9,52.8,43.9,0\n34.3,82.4,78.4,0\n15.6,35.6,57.2,0\n17.4,89.6,57.0,0\n24.3,42.7,75.4,0\n19.2,52.4,60.1,0\n21.7,35.6,76.8,0\n26.4,80.1,54.3,0\n34.2,59.2,51.1,0\n33.7,87.7,29.9,1\n16.5,38.7,26.2,1\n32.1,83.5,47.2,0\n18.4,51.6,14.1,1\n31.2,44.2,21.8,1\n20.9,69.4,59.6,0\n29.4,31.8,57.5,0\n29.4,61.5,12.0,1\n32.0,58.4,21.4,1\n34.6,32.9,53.2,0\n20.5,56.1,38.5,0\n27.6,48.3,51.2,0\n29.8,44.9,19.0,1\n16.6,85.8,70.4,0\n28.3,65.9,28.1,1\n21.4,39.2,36.2,0\n18.2,75.4,41.5,0\n26.0,52.2,51.5,0\n24.2,65.9,78.6,0\n26.5,52.8,24.7,1\n27.8,68.3,21.4,1\n25.0,70.9,20.2,1\n33.1,47.0,64.2,0\n18.6,79.9,36.8,0\n20.8,67.1,74.9,0\n22.1,66.3,14.7,1\n'
df = pd.read_csv(StringIO(csv_text))
df.to_csv(folder / "greenhouse_200.csv", index=False)
display(df.head(10))
print("Өлшемі:", df.shape)

In [ ]:
assert df.shape == (200, 4)
assert not df.isna().any().any()
assert not df.duplicated().any()
display(df.describe())
print("Санаттар саны:", df.irrigation_needed.value_counts().to_dict())
df.hist(figsize=(10, 6)); plt.tight_layout(); plt.show()

## 2. Оқыту және тест жиыны
160 жазба — оқыту, 40 — тексеру. Нәтиже бағаны X ішіне кірмейді. Scaler тек оқыту деректеріне fit жасайды, тест деректері тек transform арқылы өтеді.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
features = ["temperature_c", "air_humidity_pct", "soil_moisture_pct"]
X_train, X_test, y_train, y_test = train_test_split(df[features], df.irrigation_needed, test_size=0.2, random_state=2026, stratify=df.irrigation_needed)
print("Train:", len(X_train), "Test:", len(X_test))

## 3. Нормалау және модельді оқыту
StandardScaler: z=(x−орташа)/стандартты ауытқу. Ағаш моделіне масштабтау міндетті емес, бірақ тапсырмадағы нормалау кезеңін көрсету үшін Pipeline құрамына енгізілді.

In [ ]:
model = Pipeline([("scaler", StandardScaler()), ("classifier", DecisionTreeClassifier(max_depth=4, random_state=2026))])
model.fit(X_train, y_train)
normalized = pd.DataFrame(model.named_steps["scaler"].transform(X_train), columns=features, index=X_train.index)
display(X_train.head())
display(normalized.head())


## 4. Тест бойынша бағалау
Accuracy, precision, recall, F1 және қарапайым baseline. Матрица жолдары — нақты, бағандары — болжам. Тест нәтижесіне қарап модель параметрлерін қайта таңдамаңыз.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.dummy import DummyClassifier
pred = model.predict(X_test)
metrics = {"accuracy": accuracy_score(y_test, pred), "precision": precision_score(y_test, pred, zero_division=0), "recall": recall_score(y_test, pred, zero_division=0), "f1": f1_score(y_test, pred, zero_division=0)}
baseline=DummyClassifier(strategy="most_frequent").fit(X_train,y_train)
print(metrics)
print("Baseline accuracy:", accuracy_score(y_test,baseline.predict(X_test)))
ConfusionMatrixDisplay(confusion_matrix(y_test,pred,labels=[0,1]),display_labels=["No watering","Watering"]).plot()
plt.show()


In [ ]:
examples = pd.DataFrame([[30,40,20],[22,65,65],[32,40,38]],columns=features)
examples["prediction"] = model.predict(examples)
display(examples)

## 5. Модель, нормалау және Streamlit файлдарын сақтау
Pipeline scaler мен classifier-ді бірге сақтайды. Streamlit барлық енгізілген мәндерді сол Pipeline-ға береді.

In [ ]:
joblib.dump({"model":model,"features":features,"metrics":metrics},folder / "greenhouse_model.joblib")
requirements = "streamlit>=1.35,<2\n" + f"numpy=={np.__version__}\npandas=={pd.__version__}\nscikit-learn=={sklearn.__version__}\njoblib=={joblib.__version__}\n"
(folder / "requirements.txt").write_text(requirements)
print("Python:",sys.version)

In [ ]:
(folder / "app.py").write_text('from pathlib import Path\nimport joblib\nimport pandas as pd\nimport streamlit as st\nst.set_page_config(page_title="Диана — STEM жылыжайы", page_icon="🌱")\nst.title("🌱 STEM жылыжайы: суару болжамы")\nst.caption("Диана • 200 жасанды жазба • Google Colab → Streamlit")\nst.info("Оқу моделі шартты ережелерден жасалған деректермен оқытылған. Нақты агрономиялық ұсыныс емес.")\n@st.cache_resource\ndef load_bundle():\n    return joblib.load(Path(__file__).parent / "greenhouse_model.joblib")\ntry:\n    bundle=load_bundle()\nexcept FileNotFoundError:\n    st.error("Алдымен Colab ноутбугін орындап, greenhouse_model.joblib файлын app.py жанына салыңыз.")\n    st.stop()\nwith st.form("prediction"):\n    t=st.number_input("Температура (°C)",15.0,35.0,30.0,0.1)\n    a=st.number_input("Ауа ылғалдылығы (%)",30.0,90.0,40.0,0.1)\n    s=st.number_input("Топырақ ылғалдылығы (%)",10.0,80.0,20.0,0.1)\n    submit=st.form_submit_button("Болжам жасау")\nif submit:\n    row=pd.DataFrame([[t,a,s]],columns=bundle["features"])\n    result=int(bundle["model"].predict(row)[0])\n    if result: st.warning("💧 Суару қажет / Нужно поливать")\n    else: st.success("✅ Суару қажет емес / Поливать не нужно")\n    st.caption("Сандар сақталған StandardScaler арқылы түрлендіріліп, оқытылған модельге беріледі.")\n    with st.expander("Нормаланған мәндер"):\n        st.dataframe(pd.DataFrame(bundle["model"].named_steps["scaler"].transform(row),columns=bundle["features"]))\nst.caption("Модель бағасы: тест жиынындағы accuracy = "+str(round(bundle["metrics"]["accuracy"],3)))\n', encoding="utf-8")

In [ ]:
import shutil
shutil.make_archive("Diana_Streamlit_ready", "zip", folder)
try:
    from google.colab import files
    files.download("Diana_Streamlit_ready.zip")
except ImportError:
    print("Дайын архив: Diana_Streamlit_ready.zip")

## 6. Streamlit
Архивті ашыңыз. Windows терминалында сол қалтаға өтіп:
```
python -m pip install -r requirements.txt
python -m streamlit run app.py
```
GitHub/Streamlit үшін app.py, greenhouse_model.joblib, requirements.txt файлдарын бірге салыңыз. Оқытудағы Python major.minor нұсқасын таңдаңыз. Интернеттегі жариялау қадамын өз аккаунтыңызда орындаңыз.